# 2D Interaction Diagrams with ProLif
Please cite relevent programs incase you are going to include this notebook in your research.

### Citations:
[Prolif](https://pmc.ncbi.nlm.nih.gov/articles/PMC8466659/)
[MDAnalysis](https://pmc.ncbi.nlm.nih.gov/articles/PMC3144279/)
[OpenMM](https://pmc.ncbi.nlm.nih.gov/articles/PMC4486654/)


In Google Colab, **ProLIF** is installed and used within a virtual cloud environment to automate your structural chemistry data pipeline. Because Google Colab handles all execution on a remote Linux server, ProLIF allows you to process large molecular complexes without taxing your local computer's hardware.

In [ ]:
#@title Step 1: Installtion of the required packages
!pip install -q rdkit
!pip install -q MDAnalysis
!pip install -q biopython
!pip install -q openmm
!pip install -q pdbfixer
!pip install -q prolif

In [ ]:
#@title Step 2: Import all the necessary libraries to the workspace
import MDAnalysis as mda
from rdkit import Chem
from rdkit.Chem import rdDetermineBonds, rdMolDescriptors

from pdbfixer import PDBFixer
from openmm.app import PDBFile

from Bio.PDB import PDBParser, PDBIO, Select

import prolif as plf
from prolif.plotting.network import LigNetwork

### Below you are required to paste the information of your protein-ligand complex. Also pay attention to the print message once this code is completed. You need to find the name of your ligand (it must be there).

In [ ]:
#@title Step 3: Reading the structure
pdb_path = "/content/merged-6441.pdb"   # <-- change to your uploaded file name

u = mda.Universe(pdb_path)
print("Total atoms:", len(u.atoms))
print("Unique residue names in file:", sorted(set(u.residues.resnames)))

In [ ]:
# Pure RDKit implementation to isolate ligand and check charge simply
from rdkit import Chem

# 1. Target file path setup
pdb_path = "/content/merged-6441.pdb"

# 2. Complete PDB structure load karein (bina hydrogens ke)
complex_structure = Chem.MolFromPDBFile(pdb_path, removeHs=True, sanitize=False)

if complex_structure:
    # Sare chemical groups ko alag alag fragments mein divide karein
    fragments = Chem.GetMolFrags(complex_structure, asMols=True)

    # Sort fragments by atom count (Protein chain sab se badi hoti hai)
    fragments_sorted = sorted(fragments, key=lambda m: m.GetNumAtoms())

    # Second largest fragment ko select karna jo typical ligand size ho
    if len(fragments_sorted) > 1:
        lig_mol = fragments_sorted[-2]
    else:
        lig_mol = fragments_sorted[0]

    # 3. Formal Charge calculate aur display karein
    ligand_charge = sum(atom.GetFormalCharge() for atom in lig_mol.GetAtoms())

    print("--- Charge Check Complete ---")
    print("Isolated Ligand Atoms Count:", lig_mol.GetNumAtoms())
    print("Calculated Formal Charge is:", ligand_charge)
else:
    print("Error: PDB file read nahi ho saki. Path check karein.")

### This is the step that fixes the valence errors. Protein gets repaired with PDBFixer (missing atoms/hydrogens), and the ligand gets its bond orders re-derived from 3D geometry with RDKit's DetermineBonds (raw PDBs have no bond-order info, which is exactly why RDKit throws "explicit valence greater than permitted" errors).

In [ ]:
#@title Step 4: Structure Cleaning
LIGAND_RESNAME = "UNK"   # <-- set from the printout in step 3
LIGAND_CHARGE  = 1       # <-- set the ligand's net formal charge if it isn't neutral

# 4a. Strip alternate conformations and waters
parser = PDBParser(QUIET=True)
structure = parser.get_structure("complex", pdb_path)

class CleanSelect(Select):
    def accept_atom(self, atom):
        if atom.is_disordered() and atom.get_altloc() not in (" ", "A"):
            return False
        return True
    def accept_residue(self, residue):
        return residue.get_resname() not in ("HOH", "WAT")

io = PDBIO()
io.set_structure(structure)
io.save("/content/complex_clean.pdb", CleanSelect())

u = mda.Universe("/content/complex_clean.pdb")
protein_ag = u.select_atoms("protein")
ligand_ag  = u.select_atoms(f"resname {LIGAND_RESNAME}")

protein_ag.write("/content/protein_raw.pdb")
ligand_ag.write("/content/ligand_raw.pdb")

# 4b. Fix the protein: complete residues, add hydrogens
fixer = PDBFixer(filename="/content/protein_raw.pdb")
fixer.findMissingResidues()
fixer.missingResidues = {}     # don't model speculative loops, just fix what's present
fixer.findMissingAtoms()
fixer.addMissingAtoms()
fixer.addMissingHydrogens(7.4)

with open("/content/protein_fixed.pdb", "w") as f:
    PDBFile.writeFile(fixer.topology, fixer.positions, f)

# 4c. Fix the ligand: re-derive correct bonds/bond orders, then add hydrogens
with open("/content/ligand_raw.pdb") as f:
    ligand_block = f.read()

# removeHs=True strips any pre-existing hydrogens so DetermineBonds only
# has to reason about heavy-atom distances, which is far more reliable
lig_mol = Chem.MolFromPDBBlock(ligand_block, sanitize=False, removeHs=True, proximityBonding=False)
rdDetermineBonds.DetermineBonds(lig_mol, charge=LIGAND_CHARGE)
Chem.SanitizeMol(lig_mol)
lig_mol = Chem.AddHs(lig_mol, addCoords=True)

print("Ligand cleaned. Formula:", rdMolDescriptors.CalcMolFormula(lig_mol))

In [ ]:
#@title Step 5: Actual Calculations (Fixed for New Structure)
import MDAnalysis as mda
import prolif as plf

# 1. Initialize universe correctly
prot_u = mda.Universe("/content/protein_fixed.pdb")

# 2. Extract bonds cleanly from the fixer topology
bonds_list = [(b.atom1.index, b.atom2.index) for b in fixer.topology.bonds()]

# 3. Add bonds attribute safely using MDAnalysis topology framework
prot_u.add_TopologyAttr('bonds', bonds_list)

# 4. Standardize identifiers for ProLIF compatibility
if hasattr(prot_u.atoms, "chainIDs"):
    prot_u.atoms.chainIDs = ["A"] * len(prot_u.atoms)

if hasattr(prot_u.atoms, "segids"):
    prot_u.atoms.segments.segids = ["PROT"] * len(prot_u.atoms.segments)

# 5. Convert to ProLIF structural molecule objects
protein_mol = plf.Molecule.from_mda(prot_u)
ligand_mol  = plf.Molecule.from_rdkit(lig_mol)

# 6. Run the protein-ligand fingerprint calculations
fp = plf.Fingerprint()
fp.run_from_iterable([ligand_mol], protein_mol)

# 7. Convert results to dataframe and display table
df = fp.to_dataframe()
df

In [ ]:
#@title Step 6: Interaction Diagram in html
net = LigNetwork.from_fingerprint(fp, ligand_mol, kind="frame", frame=0)
net.display()

# also save a standalone interactive copy you can download
net.save("/content/interaction_network.html")

In [ ]:
#@title Step 7: Contact Information Retrieval
frame_data = fp.ifp[0]   # frame 0

for (lig_res, prot_res), interactions in frame_data.items():
    for interaction_name, metadata_tuple in interactions.items():
        for m in metadata_tuple:
            dist = m.get("distance")
            angle = m.get("DHA_angle", m.get("angle"))  # key name varies by interaction type
            print(f"{lig_res} - {prot_res} | {interaction_name} | distance={dist:.2f} Å"
                  + (f", angle={angle:.1f}°" if angle is not None else ""))

# **Chromium**

In Google Colab, the Chromium package refers to the open-source web browser framework (which serves as the base for Google Chrome) along with its web driver tool, chromedriver.

Since Google Colab runs on remote Linux servers without a physical screen or graphic user interface (GUI), standard browsers cannot open windows normally. The Chromium package allows Colab to run a browser entirely in the background—known as headless mode.


In [ ]:
#@title Step 8: Installing chromium
!pip install -q playwright
!playwright install --with-deps chromium

In [ ]:
#@title Step 9: Re-execution for network calculation
from playwright.async_api import async_playwright
from PIL import Image

async def html_to_png(html_path, png_path, width=1800, height=1400, scale=4):
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        page = await browser.new_page(
            viewport={"width": width, "height": height},
            device_scale_factor=scale   # scale=4 -> 4x the pixel density of a normal screen
        )
        await page.goto(f"file://{html_path}")
        await page.wait_for_timeout(5000)     # let the physics layout fully stabilize
        await page.evaluate("ifp.fit()")       # auto zoom/pan so every node is in frame
        await page.wait_for_timeout(500)
        await page.screenshot(path=png_path, full_page=True)
        await browser.close()

html_path = "/content/interaction_network.html" # Define html_path
raw_png = "/content/interaction_network_raw.png"
await html_to_png(html_path, raw_png, width=1800, height=1400, scale=4)

# Stamp DPI metadata so the file reports as 600 DPI when opened/printed
img = Image.open(raw_png)
final_png = "/content/syn-interaction_network.png"
img.save(final_png, dpi=(600, 600))

print("Final image pixel size:", img.size)

# I hope you are successfully done. Please subscribe to my youtube channel [Bioinformatics Insights](https://www.youtube.com/@Bioinformaticsinsights) for updated tutorials and research services.